# Ray Tutorial

Ray is a fast and simple framework for building and running distributed applications. This notebook walks through the basics of Ray, including how to parallelize Python functions and classes using Ray's simple API.

Based on the blog post: [Ray Tutorial](https://medium.com/codable/ray-tutorial-b3a1fb6e3601)

## Installation

First, let's install Ray using pip.

In [ ]:
#!pip install ray

## Starting Ray

To use Ray, you first need to import it and initialize it with `ray.init()`. This starts a local Ray instance. If you want to connect to an existing Ray cluster, you can pass the address of the cluster head node.

In [ ]:
import ray

ray.init()

## Remote Functions (Tasks)

In Ray, you can turn a regular Python function into a distributed task by adding the `@ray.remote` decorator. When you call a remote function, it immediately returns a future (an `ObjectRef`) rather than the result itself. The function runs asynchronously in a separate worker process.

To retrieve the result, you call `ray.get()` on the future. This blocks until the task completes and returns the result.

In [ ]:
import time

# A normal Python function
def normal_function():
    return 1

# A Ray remote function
@ray.remote
def remote_function():
    return 1

# Call a normal function — runs synchronously
result = normal_function()
print("Normal function result:", result)

# Call a remote function — returns a future immediately
future = remote_function.remote()
print("Future:", future)

# Retrieve the result
result = ray.get(future)
print("Remote function result:", result)

## Parallelizing Tasks

One of the key benefits of Ray is that you can run many tasks in parallel. Instead of waiting for each task to complete before starting the next, you can launch all tasks at once and then wait for all of them to finish.

In the example below, we compare a serial implementation versus a parallel one using Ray. Notice how the parallel version is significantly faster because all tasks run simultaneously.

In [ ]:
@ray.remote
def slow_function(i):
    time.sleep(1)
    return i

# Serial execution
start = time.time()
results = []
for i in range(4):
    result = slow_function.remote(i)
    results.append(ray.get(result))  # blocks each time
print("Serial time:", time.time() - start)
print("Serial results:", results)

# Parallel execution
start = time.time()
futures = [slow_function.remote(i) for i in range(4)]
results = ray.get(futures)  # waits for all tasks at once
print("Parallel time:", time.time() - start)
print("Parallel results:", results)

## Passing Object References

Ray allows you to pass object references (futures) directly to other remote functions. Ray will automatically handle the dependency and wait for the upstream task to complete before starting the downstream task. This makes it easy to build pipelines of tasks.

In [ ]:
@ray.remote
def add_one(value):
    time.sleep(2)  # Simulate a time-consuming task
    return value + 1

@ray.remote
def multiply_by_two(value):
    return value * 2

# Chain tasks together by passing futures directly
future_1 = add_one.remote(5)          # 5 + 1 = 6
future_2 = multiply_by_two.remote(future_1)  # 6 * 2 = 12

result = ray.get(future_2)
print("Chained result:", result)

## Remote Classes (Actors)

Ray also supports distributed stateful computation via **Actors**. An Actor is a remote class — each instance runs in its own process and maintains its own state. You define an Actor by adding the `@ray.remote` decorator to a class.

Actors are useful when you need to maintain state across multiple function calls in a distributed setting, such as a parameter server or a counter.

In [ ]:
@ray.remote
class Counter:
    def __init__(self):
        self.count = 0

    def increment(self):
        self.count += 1

    def get_count(self):
        return self.count

# Create an actor instance
counter = Counter.remote()

# Call methods on the actor
counter.increment.remote()
counter.increment.remote()
counter.increment.remote()

# Get the current count
count = ray.get(counter.get_count.remote())
print("Count:", count)

## Multiple Actors

You can create multiple Actor instances, each with its own independent state. This is useful for parallelizing stateful computations. In the example below, we create multiple counter actors and increment them independently.

In [ ]:
# Create multiple actor instances
counters = [Counter.remote() for _ in range(4)]

# Increment each counter a different number of times
for i, counter in enumerate(counters):
    for _ in range(i + 1):
        counter.increment.remote() # type: ignore

# Get all counts
counts = ray.get([counter.get_count.remote() for counter in counters])
print("Counts:", counts)

## ray.wait()

Sometimes you don't want to wait for all tasks to finish — you want to process results as they become available. `ray.wait()` allows you to do this. It takes a list of futures and returns two lists:
- **ready**: futures that are already done
- **not_ready**: futures that are still running

You can use `ray.wait()` in a loop to process results incrementally as they complete.

In [ ]:
@ray.remote
def variable_sleep(i):
    time.sleep(i * 0.5)
    return i

not_ready = [variable_sleep.remote(i) for i in range(5)]

# Process results as they become available
while not_ready:
    ready, not_ready = ray.wait(not_ready, num_returns=1)
    result = ray.get(ready[0])
    print(f"Got result: {result}")

## Putting Objects in the Object Store

Ray has a distributed in-memory object store shared among all workers. You can explicitly put an object into the object store using `ray.put()`. This is useful when you want to pass a large object to many remote functions — instead of serializing and sending the object multiple times, you store it once and pass the reference.

This avoids redundant data transfer and can significantly improve performance.

In [ ]:
import numpy as np

@ray.remote
def process_array(array, index):
    return array[index]

# Create a large array and put it in the object store once
large_array = np.arange(1000)
array_ref = ray.put(large_array)

# Pass the reference to many tasks — the array is not copied each time
futures = [process_array.remote(array_ref, i) for i in range(10)]
results = ray.get(futures)
print("Results:", results)

## Shutting Down Ray

When you're done using Ray, it's good practice to shut it down to release resources.

In [ ]:
ray.shutdown()

## Summary

In this notebook, we covered the core concepts of Ray:

| Concept | Description |
|---|---|
| `ray.init()` | Start a local Ray instance or connect to a cluster |
| `@ray.remote` on a function | Turn a function into a distributed **Task** |
| `@ray.remote` on a class | Turn a class into a distributed **Actor** |
| `.remote()` | Call a remote function or actor method asynchronously |
| `ray.get()` | Retrieve the result of one or more futures |
| `ray.wait()` | Get futures as they complete |
| `ray.put()` | Put an object in the distributed object store |
| `ray.shutdown()` | Shut down the Ray instance |

Ray makes it simple to scale Python applications from a single machine to a large cluster with minimal code changes.